In [1]:
!pip install -q datasets sentence-transformers faiss-cpu groq scikit-learn pandas numpy tqdm nltk
!pip install -q \
langchain==0.3.27 \
langchain-community==0.3.27 \
langchain-core==0.3.74 \
langchain-openai==0.3.30 \
ragas \
langchain-huggingface \
langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.5/443.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 40.9/40.9 MB 184.3 MB/s eta 0:00:01
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 179, in resolve
    self.factory.preparer.prepare_linked_requirements_more(reqs)
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/prepare.py", line 554, in prepare_linked_requirements_more
 

In [ ]:
!nvidia-smi

Fri Jul 31 12:45:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible to torch")

2.11.0+cu128
True
Tesla T4


In [3]:
from datasets import load_dataset
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)
import re, json, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
import numpy as np
import faiss, pickle, os, csv, time
from sentence_transformers import SentenceTransformer
from groq import Groq
from datetime import datetime
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    context_precision,
    context_recall,
    answer_relevancy)
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2730/2973919709.py:17: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_2730/2973919709.py:17: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics im

In [4]:
#loading dataset and creating test data
dataset=load_dataset("rungalileo/ragbench","finqa")
#print a record from train dataset
print(dataset["train"][0])
print(dataset["train"]["documents"][0])

README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

finqa/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 61.1MB            

finqa/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

finqa/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.94MB            

finqa/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

finqa/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12502 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1766 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2294 [00:00<?, ? examples/s]

{'id': 'finqa_2311', 'question': 'what is the yearly amortization rate related to the trademarks?', 'documents': ['adobe systems incorporated notes to consolidated financial statements ( continued ) we review our goodwill for impairment annually , or more frequently , if facts and circumstances warrant a review . we completed our annual impairment test in the second quarter of fiscal 2014 . we elected to use the step 1 quantitative assessment for our reporting units and determined that there was no impairment of goodwill . there is no significant risk of material goodwill impairment in any of our reporting units , based upon the results of our annual goodwill impairment test . we amortize intangible assets with finite lives over their estimated useful lives and review them for impairment whenever an impairment indicator exists . we continually monitor events and changes in circumstances that could indicate carrying amounts of our long-lived assets , including our intangible assets may 

In [5]:
#------------- Reading documents from the train dataset --------------
documents = [row["documents"] for row in dataset["test"]]
print(f"Extracted documents from {len(documents)} rows")

Extracted documents from 2294 rows


In [ ]:
#----------Writing all the documents to a json file -----------
with open("finqa.json", "w", encoding="utf=8") as f:
  json.dump(documents, f, ensure_ascii=False, indent=2)

In [ ]:
print(documents[1])
type(documents[1])

['marathon oil corporation notes to consolidated financial statements expected long-term return on plan assets 2013 the expected long-term return on plan assets assumption for our u.s . funded plan is determined based on an asset rate-of-return modeling tool developed by a third-party investment group which utilizes underlying assumptions based on actual returns by asset category and inflation and takes into account our u.s . pension plan 2019s asset allocation . to determine the expected long-term return on plan assets assumption for our international plans , we consider the current level of expected returns on risk-free investments ( primarily government bonds ) , the historical levels of the risk premiums associated with the other applicable asset categories and the expectations for future returns of each asset class . the expected return for each asset category is then weighted based on the actual asset allocation to develop the overall expected long-term return on plan assets assu

list

In [ ]:
#-------------Fixed Size Chunking---------------#
def fixed_size_chunk(text, chunk_size=512):
  return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [ ]:
#-------------- Recursive Character Chunking ----------------#
def recursive_chunk(text, chunk_size=512, chunk_overlap=50):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      separators=["\n\n", "\n", ". ", " ", ""])
  return splitter.split_text(text)

In [ ]:
#--------------- Sentence Chunking-----------
def sentence_chunk(text, sentences_per_chunk=5):
  sentences = sent_tokenize(text)
  return [" ".join(sentences[i:i+sentences_per_chunk]) for i in range(0, len(sentences), sentences_per_chunk)]


In [ ]:
from typing_extensions import NoExtraItems
#-------------- Table Aware Chunking --------------
def _is_table_line(l):
  stripped = l.strip()
  if not stripped: return False
  if "|" in l: return True
  if re.search(r"(\t.*){2,}", l): return True
  if len(re.findall(r"\d+[\.,]?\d*",l)) >= 3 and len(l.split()) <=12:
    return True
  if stripped.startswith("[[") and stripped.endswith("]]"):
    return True
  if l.count("\",") >=4 and ("[" in l or "]" in l):
    return True
  return False


def _extract_nested_list_tables(text):
  """
  find subsctrings that look like [[...],[...]] json nested lists inside text.
  Returns list of (start, end, parsed_rows or noe, raw_substring).
  """
  results = []
  pattern = re.compile(r"\[\[.*?\]\]", re.DOTALL)
  for m in pattern.finditer(text):
    raw = m.group(0)
    parsed = None
    try:
      candidate = json.loads(raw)
      if (isinstance(candidate, list) and candidate
          and all(isinstance(r, list) for r in candidate)):
          parsed = candidate
    except Exception:
      parsed = None
    results.append((m.start(), m.end(), parsed, raw))
  return results


def _format_table_rows(rows):
  """ Turns a list-of-lists table into a marckdown-style pipe table. """
  norm = [[str(c).strip() for c in r] for r in rows]
  width = max(len(r) for r in norm)
  norm = [r + [""] * (width - len(r)) for r in norm]

  header = norm[0]
  body = norm[1:] if len(norm) > 1 else []
  lines = ["| " + " | ".join(header) + " |",
           "| " + " | ".join(["---"]*width) + " |"]
  for r in body:
    lines.append("| " + " | ".join(r) + " |")
  return "\n".join(lines)


def _fixed_size_chunk(text, chunk_size):
  text = text.strip()
  if not text:
    return []
  return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


def table_aware_chunk(text, chunk_size=512):
  """extracts json nested-lists tables
  detects pipes/tab
  chunks surrounding prose with fixed size splitting
  """

  if not text:
    return []

  chunks = []

  nested = _extract_nested_list_tables(text)
  if nested:
    cursor = 0
    for start, end, parsed, raw in nested:
      prose = text[cursor:start]
      if prose.strip():
        chunks.extend(_chunk_prose_and_line_tables(prose, chunk_size))
      if parsed:
        chunks.append(_format_table_rows(parsed))
      else:
        chunks.append(raw.strip())
      cursor = end
    tail = text[cursor:]
    if tail.strip():
      chunks.extend(_chunk_prose_and_line_tables(tail, chunk_size))
    return [c for c in chunks if c and c.strip()]
  return _chunk_prose_and_line_tables(text, chunk_size)


def _chunk_prose_and_line_tables(text, chunk_size):
  lines = text.split("\n")
  out, buffer, table_buffer, in_table = [], [], [], False

  def flush_prose():
    if buffer:
      out.extend(_fixed_size_chunk(" ".join(buffer), chunk_size))
      buffer.clear()

  def flush_table():
    nonlocal in_table
    if table_buffer:
      out.append("\n".join(table_buffer))
      table_buffer.clear()
    in_table = False

  for line in lines:
    if _is_table_line(line):
      if not in_table:
        flush_prose()
      in_table = True
      table_buffer.append(line)
    else:
      if in_table:
        flush_table()
      if line.strip():
        buffer.append(line)
  flush_prose()
  flush_table()
  return out

In [ ]:
#------------------ sliding window--------------
def sliding_window_chunk(text, window_size=512, step=250):
  return [text[i:i+window_size] for i in range(0, max(len(text)-window_size+1, 1), step)]

In [ ]:
def structure_aware_chunk(text, max_chunk_size=800):
  #split on headings / blank lines / bullet marks
  section_pattern = re.compile(
      r"(?:\n\s*\n)|(?:#{1,6}\s|\d+\. \s|[-*]\s|[A-Z][A-Z\s]{3, }$)",
      re.MULTILINE
  )
  sections = [s.strip() for s in section_pattern.split(text) if s and s.strip()]
  chunks, current = [], ""
  for sec in sections:
    if len(current) + len(sec) + 1 <= max_chunk_size:
      current = (current + "\n" + sec).strip()
    else:
      if current:
        chunks.append(current)
      if len(sec) > max_chunk_size:
        chunks.extend(fixed_size_chunk(sec, max_chunk_size))
        current = ""
      else:
        current = sec
  if current:
    chunks.append(current)
  return chunks

In [ ]:
import re
#----------- Main chunking Dispatcher ------------
def chunk_text(text, config):
  """
  config = {"strategy": "fixed" | "recursive" | "sentence" | "sliding" | "structure",
  "params": {"chunk_size": int, "chunk_overlap": int, "sentences_per_chunk": int,
  "window_size": int, "step": int, "max_chunk_size": int}
  }
  """
  strategy = config["strategy"]
  params = config["params"]
  dispatch = {
      "fixed": fixed_size_chunk,
      "recursive": recursive_chunk,
      "sentence": sentence_chunk,
      "sliding": sliding_window_chunk,
      "structure": structure_aware_chunk,
      "table": table_aware_chunk
  }
  if strategy not in dispatch:
    raise ValueError(f"Invalid chunking strategy: {strategy}")
  return dispatch[strategy](text, **params)

def chunk_documents(documents, config):
  # Join list of strings into a single string before chunking
  return [{"doc_index":i, "chunks": chunk_text(" ".join(doc), config)}
          for i, doc in enumerate(documents)]

In [ ]:
#---------Embedding chunks and building FAISS ----------------

#---------------- Flatten Chunks -----------------------------
def flatten_chunks(chunked_results):
  """
  chunked_results: dict like {strategy_name: [{"doc_index": int, "chunks": [str]}]}
  retunrs: (text, metadatas)
  """

  texts, metas = [], []
  for strategy, docs in chunked_results.items():
    for d in docs:
      for ci, chunk in enumerate(d["chunks"]):
        if chunk and chunk.strip():
          texts.append(chunk)
          metas.append({
              "strategy": strategy,
              "doc_index": d["doc_index"],
              "chunk_index": ci
          })
  return texts, metas

#----------- Embed -------------
def embed_texts(text, model_name="BAAI/bge-small-en-v1.5", # Corrected model name
                batch_size=64):
  print(f" Embedding model used =====> {model_name}")
  model = SentenceTransformer(model_name)
  embeddings = model.encode(
      text,
      batch_size=batch_size,
      show_progress_bar=True,
      convert_to_numpy=True,
      normalize_embeddings=True
  )
  return embeddings.astype("float32"), model

#------------ build FAISS Index ----------------
def build_faiss_index(embeddings):
  dim=embeddings.shape[1]
  index = faiss.IndexFlatIP(dim)
  index.add(embeddings)
  print(f"FAISS index build: {index.ntotal} vectors, dim={dim}")
  return index


#----------- save index.faiss --------------------------
def save_index(index, metas, texts, strategy, out_dir="faiss.store"):
  os.makedirs(out_dir, exist_ok=True)
  safe = strategy.strip().replace(" ", "_").lower()
  index_path = os.path.join(out_dir, f"index_test_bge_{safe}.faiss")
  meta_path  = os.path.join(out_dir, f"meta_test_bge_{safe}.pkl")

  faiss.write_index(index, index_path)
  with open(meta_path, "wb") as f:
    pickle.dump({"metas": metas, "texts": texts, "strategy": safe}, f)
  print(f"Saved: {index_path} ({index.ntotal} vectors)")
  print(f"Saved: {meta_path}")
  return index_path, meta_path

In [6]:
#----------- Retrieval/ search from index.faiss ----------------------

# Simple in memory cache
_INDEX_CACHE = {}
_MODEL_CACHE = {}

#all-MiniLM-L6-v2
#BAAI/bge-small-en-v1.5
def _get_model(model_name="all-MiniLM-L6-v2"):
  if model_name not in _MODEL_CACHE:
    _MODEL_CACHE[model_name] = SentenceTransformer(model_name)
  return _MODEL_CACHE[model_name]

#------------Load index ------------------------------
DRIVE_FAISS_DIR = "/content/drive/MyDrive/faiss_store"

def load_index(strategy, out_dir=DRIVE_FAISS_DIR, index_path="",meta_path=""):
  out_dir = "/content/drive/MyDrive/faiss_store"
  safe = strategy.strip().replace(" ", "_").lower()
  if safe in _INDEX_CACHE:
    return _INDEX_CACHE[safe]

  print("======= LOAD Index =======")
  index_path = os.path.join(out_dir,index_path)
  meta_path  = os.path.join(out_dir,meta_path)
  print(f"out_dir----->{out_dir}")
  print(f"index_path----->{index_path}")
  print(f"meta_path----->{meta_path}")

  if not (os.path.exists(index_path) and os.path.exists(meta_path)):
    raise FileNotFoundError(
        f"No Saved index for strategy '{strategy}' in '{out_dir}'. "
        f"Expected Index path : {index_path} and {meta_path}")

  index = faiss.read_index(index_path)
  with open(meta_path, "rb") as f:
    store = pickle.load(f)

  bundle = {"index": index, "metas": store["metas"], "texts": store["texts"]}
  _INDEX_CACHE[safe] = bundle
  return bundle

#BAAI/bge-reranker-base
#ms-marco-MiniLM-L-6-v2
#----------- Searching Index --------------------
def search(query, strategy, k=5, out_dir="faiss_store",
           rerank=False, rerank_model="BAAI/bge-reranker-base",
           rerank_candidates = 10,index_path="",meta_path="",
                embedding_model=""):
  """
  if rerank=true, fetch, rerank_candidate from FAISS, then cross endocode and retunrs top 'k'
  """

  bundle = load_index(strategy, out_dir=out_dir,index_path=index_path,meta_path=meta_path)
  index, metas, texts = bundle["index"], bundle["metas"], bundle["texts"]

  print("========== Search=========")
  print(f"Emedding Model--->{embedding_model}")
  print(f"Reranking Model--->{rerank_model}")
  print(f"Rerank--->{rerank}")

  model = _get_model(embedding_model)
  q_emb = model.encode([query],
                       normalize_embeddings=True,
                       convert_to_numpy=True
                       ).astype("float32")

  scores, idxs = index.search(q_emb, k)

  results = []
  for score, i in zip(scores[0], idxs[0]):
    if i == -1:
      continue
    results.append({
        "score": float(score),
        "text": texts[i],
        **metas[i],
    })

    if rerank:
      results = rerank_hits(query, results, top_k=k, model_name=rerank_model)
  return results


In [7]:
#--------------- ReRanking ------------------
_RERANKER_CACHE = {}

def _get_reranker(model_name="BAAI/bge-reranker-base"):
  if model_name not in _RERANKER_CACHE:
    _RERANKER_CACHE[model_name] = CrossEncoder(model_name)
  return _RERANKER_CACHE[model_name]

def rerank_hits(query, hits, top_k=5, model_name="BAAI/bge-reranker-base"):
  if not hits:
    return hits
  #model_name="BAAI/bge-reranker-base"
  print(f"Reranking model name ====> {model_name}")
  reranker = _get_reranker(model_name)
  pairs = [(query, h["text"]) for h in hits]
  scores = reranker.predict(pairs, show_progress_bar=False)

  for h, s in zip(hits, scores):
    h["retriever_score"] = h.get("score")
    h["reranker_score"] = float(s)
    h["score"] = float(s)

  hits_sorted = sorted(hits, key=lambda x: x["reranker_score"], reverse=True)
  return hits_sorted[:top_k]


In [8]:
#building Prompt
def build_prompt(question, hits, max_context_chars=6000, system_instructions=None):

    context_parts, used = [], 0

    for i, h in enumerate(hits, start=1):
      tag = f"[S{i} | doc={h.get('doc_index', '?')}] chunk={h.get('chunk_index','?')} score={h['score']:.3f}]"
      block = f"{tag}\n{h['text'].strip()}\n"
      if used + len(block) > max_context_chars:
        break
      context_parts.append(block)
      used += len(block)

    context = "\n".join(context_parts) if context_parts else "No context available"
    user_prompt = (
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"Instructions:\n"
        f"- Answer using only the sources above.\n"
        f"- Cite the source number(s) inline for every factual claim, e.g. \"[1]\".\n"
        f"- Show any calculation steps explicitly before stating the final number.\n"
        f"- Give a clear, direct final answer as the last line, prefixed with 'Answer:'."
    )

    system_instruction = system_instructions or (
        "You are a financial question answering assistant.\n\n"
        "Answer ONLY using the provided context — never use outside knowledge.\n\n"
        "You may:\n"
        "- compare values across sources\n"
        "- compute ratios or derived metrics\n"
        "- perform arithmetic\n"
        "- infer values directly derivable from the context\n\n"
        "Always show your reasoning before giving the final answer, and cite the "
        "source number(s) you used for each claim.\n\n"
        "If the answer cannot be determined exactly from the context:\n"
        "1. State what information is available.\n"
        "2. Explain specifically why the answer cannot be determined.\n"
        "3. Note any inconsistencies or missing data.\n"
        "4. End with a short conclusion (e.g. 'Answer: Cannot be determined from context')."
    )

    full = f"<<SYSTEM>>\n{system_instruction}\n\n<<USER>>\n{user_prompt}"
    return {"system": system_instruction, "user": user_prompt, "full": full}

In [ ]:
#-------------- GROQ LLM to generate the answer -----------
_GROQ_CLIENT = None

def _get_groq_client():
  global _GROQ_CLIENT
  if _GROQ_CLIENT is None:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # Set your API key as environment variable
    if not GROQ_API_KEY:
      raise ValueError("Please set GROQ_API_KEY environment variable")
    _GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
  return _GROQ_CLIENT


def answer_with_groq(query, strategy, k=5,
                     model = "llama-3.1-8b-instant",
                     temperature = 0.2,
                     max_tokens=512,
                     rerank=False, rerank_candidates = 10,index_path="",meta_path="",
                embedding_model="",
                rerank_model=""):
  print("========answer_with_groq=======")
  print(f"LLm Model used ====> {model}")
  hits = search(query, strategy, k=k,
                rerank=rerank, rerank_candidates=rerank_candidates,index_path=index_path,meta_path=meta_path,
                embedding_model=embedding_model,
                rerank_model=rerank_model)
  if not hits:
    return {"answer": "No answer found", "hits": hits}
  prompt = build_prompt(query, hits)

  client = _get_groq_client()
  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": prompt["system"]},
          {"role": "user", "content": prompt["user"]}
      ],
      temperature=temperature,
      max_tokens=max_tokens
  )
  answer = response.choices[0].message.content
  return {"answer": answer, "hits": hits, "prompt":prompt}

In [ ]:
#-------------- FinMA-7B-NLP (ChanceFocus) local LLM to generate the answer -----------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

_FINMA_MODEL = None
_FINMA_TOKENIZER = None

def _get_finma(model_name="ChanceFocus/finma-7b-nlp"):
  global _FINMA_MODEL, _FINMA_TOKENIZER
  if _FINMA_MODEL is None:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    _FINMA_TOKENIZER = AutoTokenizer.from_pretrained(model_name)
    _FINMA_MODEL = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    _FINMA_MODEL.eval()
  return _FINMA_MODEL, _FINMA_TOKENIZER


def answer_with_finma(query, strategy, k=5,
                       model_name="ChanceFocus/finma-7b-nlp",
                       max_new_tokens=512,
                       temperature=0.2,
                       rerank=False, rerank_candidates=10):
  # Same retrieval + prompt-building path as answer_with_groq()
  hits = search(query, strategy, k=k,
                rerank=rerank, rerank_candidates=rerank_candidates)
  if not hits:
    return {"answer": "No answer found", "hits": hits}

  prompt = build_prompt(query, hits)
  model, tokenizer = _get_finma(model_name)

  # prompt["full"] already contains the real system instructions + context + question
  inputs = tokenizer(
      prompt["full"], return_tensors="pt",
      truncation=True, max_length=4096
  ).to(model.device)

  with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

  full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
  input_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
  answer = full_text[len(input_text):].strip()

  return {"answer": answer, "hits": hits, "prompt": prompt}

In [ ]:
#cell no 23
from huggingface_hub import file_exists
import math
#----------- Calculating RAGAS Metrics-----------------

#---------- Mount Google Drive

#----------- RAGAS Metrics -----------------

def _to_float(value):
  if value is None:
    return None
  try:
    f = float(value)
    if math.isnan(f):
      return None
    return f
  except (ValueError, TypeError):
    return None
#------------ Judge LLM + embeddings for RAGAS
def _build_judge(model="llama-prompt-guard-2-86m",
                 embed_model="all-MiniLM-L6-v2"):
  groq_api_key = os.getenv("GROQ_API_KEY")  # Set your API key as environment variable
  if not groq_api_key:
    raise ValueError("Please set GROQ_API_KEY environment variable")
  judge_llm = ChatGroq(
      model=model,
      temperature = 0.0,
      api_key=groq_api_key,
      n=1
  )
  judge_embeddings = HuggingFaceEmbeddings(model_name=embed_model)
  return judge_llm, judge_embeddings


# ----------------- CSV Logging ----------
FLOAT_COLUMNS = ["relevance", "utilization", "adherence", "completeness"]

CSV_COLUMNS = [
    "run_id", "question",
    "embedding_model", "llm_model", "judge_llm_model",
    "chunking_strategy", "chunk_size", "top_k", "reranking", "rerank_model",
    "relevance", "utilization", "adherence", "completeness",
    "answer", "ground_truth"
]

def append_run_to_csv(row, csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv"):
  #csv_path = "/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv"
  print(f"CSV Path ====> {csv_path}")
  file_exists = os.path.isfile(csv_path)
  full_row = {}
  for c in CSV_COLUMNS:
    v = row.get(c)
    if c in FLOAT_COLUMNS:
      fv = _to_float(v)
      full_row[c] = f"{fv:.6f}" if fv is not None else ""
    else:
      full_row[c] = v
  with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    if not file_exists:
      writer.writeheader()
    writer.writerow(full_row)


#--------- End to End: run RAG + Evaluate + Log

def evaluate_and_log(
    question, dataset_train, dataset_row_index,
    strategy, config, k=5,
    llm_model="llama-3.3-70b-versatile",
    judge_model="llama-3.3-70b-versatile",
    csv_path = "rag_finqa_RMSE1_runs.csv",):
  """
  Runs your rag pipeline (answer_with_groq)
  pulls ground truth from dataset_train
  and evaluates RAGAS Metrics
  Appends to csv_path
  """

  #Run RAG with Groq
  rag_out = answer_with_groq(
      query=question, strategy=strategy, k=k,
      model=llm_model,
      rerank=bool(config.get("reranking", False)),
      rerank_candidates=config.get("rerank_candidates", 10),
                embedding_model="",
                rerank_model="",)

  #Run Rag with Finma
  #rag_out = answer_with_finma(query=question, strategy=strategy, k=k,
   #                    model_name="ChanceFocus/finma-7b-nlp",
    #                   max_new_tokens=512,
      #                 temperature=0.2,
       #                rerank=bool(config.get("reranking", False)), rerank_candidates=config.get("rerank_candidates", 20),)



  resp = evaluate_trace(question, rag_out)
  row = export_trace_to_csv(resp)
  return row

In [ ]:
#Computing metrics
from huggingface_hub import file_exists
import math
import re
import json
import time


ABBR = {"eg", "ie", "vs", "fig", "no", "cf", "al", "approx", "dr", "etc",
        "eq", "ref", "spp", "sp", "st", "mr", "ms"}


def split_sentences(text):
    text = text.replace("\n", " ").strip()
    if not text:
        return []
    try:
        return [s.strip() for s in sent_tokenize(text) if s.strip()]
    except Exception:
        pass
    # regex fallback with light abbreviation protection
    parts = re.split(r"(?<=[.!?])\s+", text)
    out, buf = [], ""
    for p in parts:
        buf = (buf + " " + p).strip() if buf else p
        toks = buf.split()
        last = re.sub(r"[^A-Za-z]", "", toks[-1]).lower() if toks else ""
        if last in ABBR:
            continue
        out.append(buf)
        buf = ""
    if buf:
        out.append(buf)
    return [s.strip() for s in out if s.strip()]


def key_context_sentences(retrieved_docs):
    """retrieved_docs: list of hit dicts with a 'text' field -- same shape
    as Mahi's `rag_out["hits"]`, so no adapter needed."""
    keyed, context_text = [], ""
    letters = "abcdefghijklmnopqrstuvwxyz"
    for doc_idx, doc in enumerate(retrieved_docs):
        context_text += f"\nDocument {doc_idx}:\n"
        for sent_idx, sent in enumerate(split_sentences(doc["text"])):
            key = f"{doc_idx}{letters[sent_idx]}" if sent_idx < len(letters) else f"{doc_idx}{sent_idx}"
            keyed.append({"key": key, "sentence": sent})
            context_text += f"{key}: {sent}\n"
    return keyed, context_text


def key_response_sentences(response_text):
    letters = "abcdefghijklmnopqrstuvwxyz"
    keyed, text_keyed = [], ""
    for i, sent in enumerate(split_sentences(response_text)):
        key = letters[i] if i < len(letters) else str(i)
        keyed.append({"key": key, "sentence": sent})
        text_keyed += f"{key}: {sent}\n"
    return keyed, text_keyed


def judge_rag_response(question, context_for_judge, response_for_judge,
                        judge_model="llama-3.3-70b-versatile"):
    prompt = f"""
You are an expert RAG evaluator.
You will be given:
1. A question
2. Retrieved context sentences with sentence keys
3. A generated response split into sentence keys

Your job is to evaluate the response using only the provided context.
Return ONLY valid JSON with the following fields:
{{
  "all_relevant_sentence_keys": ["context keys relevant to the question"],
  "overall_supported": true or false,
  "sentence_support_information": [
    {{"response_sentence_key": "a", "fully_supported": true or false,
      "supporting_sentence_keys": ["context keys that support this response sentence"]}}
  ],
  "all_utilized_sentence_keys": ["context keys actually used to support the response"]
}}

Important rules:
- Use only the provided context.
- Do not invent sentence keys.
- A relevant sentence helps answer the question.
- A utilized sentence directly supports the generated response.
- overall_supported is true only if every response sentence is fully supported by the context.
- Return JSON only. No markdown, no explanations.

Question:
{question}

Retrieved Context:
{context_for_judge}

Generated Response:
{response_for_judge}
"""
    groq_api_key = os.getenv("GROQ_API_KEY")  # Set your API key as environment variable
    if not groq_api_key:
      raise ValueError("Please set GROQ_API_KEY environment variable")
    client = Groq(api_key=groq_api_key)
    resp = client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": "You are a strict RAG evaluation judge. Return only valid JSON."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0, max_tokens=700,
    )
    return resp.choices[0].message.content.strip()


def parse_json_safely(text):
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None


def compute_trace_metrics(judge_output, total_context_sentence_count):
    relevant = set(judge_output.get("all_relevant_sentence_keys", []))
    utilized = set(judge_output.get("all_utilized_sentence_keys", []))

    if total_context_sentence_count == 0:
        relevance_score = utilization_score = 0.0
    else:
        relevance_score = len(relevant) / total_context_sentence_count
        utilization_score = len(utilized) / total_context_sentence_count

    completeness_score = None if len(relevant) == 0 else len(relevant & utilized) / len(relevant)

    adherence_score = bool(judge_output.get("overall_supported", False))
    info = judge_output.get("sentence_support_information", [])
    if info:
        adherence_continuous = sum(1 for s in info if s.get("fully_supported", False)) / len(info)
    else:
        adherence_continuous = float(adherence_score)

    return {
        "pred_relevance_score": relevance_score,
        "pred_utilization_score": utilization_score,
        "pred_completeness_score": completeness_score,
        "pred_adherence_score": adherence_score,
        "pred_adherence_continuous": adherence_continuous,
    }


def evaluate_trace(question, rag_out, judge_model="llama-3.3-70b-versatile"):
    answer = rag_out["answer"]
    retrieved_docs = rag_out["hits"]  # list of dicts with "text" -- unchanged shape

    keyed_context, context_for_judge = key_context_sentences(retrieved_docs)
    keyed_response, response_for_judge = key_response_sentences(answer)

    print("========evaluate_trace=======")
    print(f"judge_model used ====> {judge_model}")

    judge_output = parse_json_safely(
        judge_rag_response(question, context_for_judge, response_for_judge, judge_model=judge_model)
    )
    if judge_output is None:
        raise ValueError("Judge output could not be parsed as JSON")

    total_ctx = len(keyed_context)
    m = compute_trace_metrics(judge_output, total_ctx)

    scores = {
        "relevance": m["pred_relevance_score"],
        "utilization": m["pred_utilization_score"],
        "adherence": m["pred_adherence_continuous"],   # continuous, for RMSE
        "adherence_bool": m["pred_adherence_score"],    # boolean, for AUROC
        "completeness": m["pred_completeness_score"],
    }
    return scores


FLOAT_COLUMNS = ["relevance", "utilization", "adherence", "completeness",
                  "gold_relevance", "gold_utilization", "gold_adherence", "gold_completeness"]

CSV_COLUMNS = [
    "run_id", "question",
    "embedding_model", "llm_model", "judge_llm_model",
    "chunking_strategy", "chunk_size", "top_k", "reranking", "rerank_model",
    "relevance", "utilization", "adherence", "completeness",
    "gold_relevance", "gold_utilization", "gold_adherence", "gold_completeness",
    "answer", "ground_truth"
]


def _to_float(value):
    if value is None:
        return None
    try:
        f = float(value)
        if math.isnan(f):
            return None
        return f
    except (ValueError, TypeError):
        return None


def append_run_to_csv(row, csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv"):
    print(f"CSV Path ====> {csv_path}")
    file_exists_flag = os.path.isfile(csv_path)
    full_row = {}
    for c in CSV_COLUMNS:
        v = row.get(c)
        if c in FLOAT_COLUMNS:
            fv = _to_float(v)
            full_row[c] = f"{fv:.6f}" if fv is not None else ""
        else:
            full_row[c] = v
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists_flag:
            writer.writeheader()
        writer.writerow(full_row)


def export_trace_to_csv(question, embedding_model, llm_model, judge_model, strategy,
                         reranking, rerank_model, answer, ground_truth, scores,
                         gold_scores=None):
    row = {
        "run_id": f"run_{int(time.time())}",
        "question": question,
        "embedding_model": embedding_model,
        "llm_model": llm_model,
        "judge_llm_model": judge_model,
        "chunking_strategy": strategy,
        "chunk_size": 1024,
        "top_k": 5,
        "reranking": reranking,
        "rerank_model": rerank_model,
        "answer": answer,
        "ground_truth": ground_truth,
        **scores,
    }
    if gold_scores:
        row.update(gold_scores)
    append_run_to_csv(row, csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv")
    return row


def run_batch_eval(
        dataset, n, llm_model, judge_model, embedding_model, strategy, reranking, rerank_model,
        index_path="", meta_path="", k=5,
        csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv",
        sleep_between_calls=60):
    test_set = dataset["test"].select(range(n)) if n < len(dataset["test"]) else dataset["test"]
    results = []

    for i, example in enumerate(test_set):
        print(f"Running example {i+1}")
        question = example["question"]
        ground_truth = example.get("ground_truth") or example.get("answer")

        # NEW: RAGBench's own gold TRACe scores for this example
        gold_scores = {
            "gold_relevance": example.get("relevance_score"),
            "gold_utilization": example.get("utilization_score"),
            "gold_completeness": example.get("completeness_score"),
            "gold_adherence": example.get("adherence_score"),
        }

        try:
            rag_out = answer_with_groq(
                query=question, strategy=strategy, k=k,
                model=llm_model,
                temperature=0.2,
                max_tokens=512,
                rerank=reranking, rerank_candidates=10, index_path=index_path, meta_path=meta_path,
                embedding_model=embedding_model,
                rerank_model=rerank_model)
            scores = evaluate_trace(question, rag_out, judge_model=judge_model)
            answer = rag_out["answer"]
            row = export_trace_to_csv(question, embedding_model, llm_model, judge_model, strategy,
                                       reranking, rerank_model, answer, ground_truth, scores,
                                       gold_scores=gold_scores)
            results.append(row)

        except Exception as e:
            print(f"⚠️ Failed on question {i}: {question[:80]}... -> {e}")
            results.append({"question": question, "error": str(e)})

        if sleep_between_calls:
            time.sleep(sleep_between_calls)

        if (i + 1) % 10 == 0:
            print(f"Completed {i+1}/{len(test_set)}")

    return results


import numpy as np
import pandas as pd


def rmse_masked(dfr, predcol, goldcol):
    p = pd.to_numeric(dfr[predcol], errors="coerce")
    g = pd.to_numeric(dfr[goldcol], errors="coerce")
    m = p.notna() & g.notna()
    if not m.any():
        return float("nan"), 0
    return float(np.sqrt(np.mean((p[m] - g[m]) ** 2))), int(m.sum())


def compute_trace_rmse(csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv"):
    df = pd.read_csv(csv_path)
    summary = {"n_rows": len(df)}

    for metric in ["relevance", "utilization", "completeness"]:
        r, n = rmse_masked(df, metric, f"gold_{metric}")
        summary[f"rmse_{metric}"] = r
        summary[f"rmse_{metric}_n"] = n

    # Adherence's gold label is boolean -> AUROC, not RMSE (same reasoning
    # Harshita used: continuous predicted score vs binary gold label).
    try:
        from sklearn.metrics import roc_auc_score
        gold_adh = pd.to_numeric(df["gold_adherence"], errors="coerce")
        pred_adh = pd.to_numeric(df["adherence"], errors="coerce")
        mask = gold_adh.notna() & pred_adh.notna()
        summary["auroc_adherence"] = (
            float(roc_auc_score(gold_adh[mask], pred_adh[mask])) if mask.sum() > 1 else None
        )
    except Exception as e:
        summary["auroc_adherence"] = None
        summary["auroc_error"] = str(e)

    print(json.dumps(summary, indent=2))
    return summary


# ---- Run it ----
# trace_rmse_summary = compute_trace_rmse()

In [32]:
# --- usage ---
results = run_batch_eval(
    dataset=dataset,
    n=5,                       # number of questions to run
    llm_model="qwen/qwen3.6-27b",
    judge_model="llama-3.3-70b-versatile",
    embedding_model="all-MiniLM-L6-v2",
    strategy="table",
    reranking= True,
    rerank_model="ms-marco-MiniLM-L-6-v2",index_path="index_test_MiniLM_table.faiss",meta_path="meta_test_MiniLM_table.pkl",
    k=5,
    csv_path="/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv",
    sleep_between_calls=70,
)

Running example 1
========answer_with_groq=======
LLm Model used ====> qwen/qwen3.6-27b
========== Search=========
Emedding Model--->all-MiniLM-L6-v2
Reranking Model--->ms-marco-MiniLM-L-6-v2
Rerank--->True
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
========evaluate_trace=======
judge_model used ====> llama-3.3-70b-versatile
CSV Path ====> /content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv
Running example 2
========answer_with_groq=======
LLm Model used ====> qwen/qwen3.6-27b
========== Search=========
Emedding Model--->all-MiniLM-L6-v2
Reranking Model--->ms-marco-MiniLM-L-6-v2
Rerank--->True
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-MiniLM-L-6-v2
Reranking model name ====> ms-marco-M

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import time

# Define all 14 run configurations as a list of dictionaries
experiments = [
    {
        "strategy": "recursive",
        "index_path": "index_test_bge_recursive.faiss",
        "meta_path": "meta_test_bge_recursive.pkl",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "llm_model": "qwen/qwen3.6-27b",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": "BAAI/bge-reranker-base",
    },
    {
        "strategy": "sliding",
        "index_path": "index_test_bge_sliding.faiss",
        "meta_path": "meta_test_bge_sliding.pkl",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "llm_model": "llama-3.1-8b-instant",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": "BAAI/bge-reranker-base",
    },
    {
        "strategy": "table",
        "index_path": "index_test_bge_table.faiss",
        "meta_path": "meta_test_bge_table.pkl",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "llm_model": "qwen/qwen3.6-27b",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": "BAAI/bge-reranker-base",
    },
    {
        "strategy": "table",
        "index_path": "index_test_MiniLM_table.faiss",
        "meta_path": "meta_test_MiniLM_table.pkl",
        "embedding_model": "all-MiniLM-L6-v2",
        "llm_model": "llama-3.1-8b-instant",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": "ms-marco-MiniLM-L-6-v2",
    },
    {
        "strategy": "table",
        "index_path": "index_test_MiniLM_table.faiss",
        "meta_path": "meta_test_MiniLM_table.pkl",
        "embedding_model": "all-MiniLM-L6-v2",
        "llm_model": "llama-3.1-8b-instant",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": None,
    },
    {
        "strategy": "table",
        "index_path": "index_test_bge_table.faiss",
        "meta_path": "meta_test_bge_table.pkl",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "llm_model": "llama-3.1-8b-instant",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": None,
    },
    {
        "strategy": "table",
        "index_path": "index_test_bge_table.faiss",
        "meta_path": "meta_test_bge_table.pkl",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "llm_model": "qwen/qwen3.6-27b",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": None,
    },
    {
        "strategy": "table",
        "index_path": "index_test_MiniLM_table.faiss",
        "meta_path": "meta_test_MiniLM_table.pkl",
        "embedding_model": "all-MiniLM-L6-v2",
        "llm_model": "qwen/qwen3.6-27b",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": "ms-marco-MiniLM-L-6-v2",
    },
    {
        "strategy": "table",
        "index_path": "index_test_MiniLM_table.faiss",
        "meta_path": "meta_test_MiniLM_table.pkl",
        "embedding_model": "all-MiniLM-L6-v2",
        "llm_model": "qwen/qwen3.6-27b",
        "judge_model": "llama-3.3-70b-versatile",
        "reranker": None,
    },
]

# Shared parameters across all evaluations
COMMON_PARAMS = {
    "dataset": dataset,
    "n": 5,
    "k": 5,
    "csv_path": "/content/drive/MyDrive/Rag_Eval/rag_finqa_RMSE1_runs.csv",
    "sleep_between_calls": 70,
}

# Loop and run
for idx, config in enumerate(experiments, start=1):
    print(f"\n--- Running Config {idx}/{len(experiments)} ---")

    # Check if a reranker is provided
    reranker = config.get("reranker")
    is_reranking = bool(reranker)
    rerank_model = reranker if is_reranking else "BAAI/bge-reranker-base"

    try:
        run_batch_eval(
            strategy=config["strategy"],
            index_path=config["index_path"],
            meta_path=config["meta_path"],
            embedding_model=config["embedding_model"],
            llm_model=config["llm_model"],
            judge_model=config["judge_model"],
            reranking=is_reranking,
            rerank_model=rerank_model,
            **COMMON_PARAMS
        )
        print(f"Completed Config {idx}")
    except Exception as e:
        print(f"Error in Config {idx}: {e}")


--- Running Config 1/9 ---
Running example 1
========answer_with_groq=======
LLm Model used ====> qwen/qwen3.6-27b
========== Search=========
Emedding Model--->BAAI/bge-small-en-v1.5
Reranking Model--->BAAI/bge-reranker-base
Rerank--->True
Reranking model name ====> BAAI/bge-reranker-base
Reranking model name ====> BAAI/bge-reranker-base
Reranking model name ====> BAAI/bge-reranker-base
Reranking model name ====> BAAI/bge-reranker-base
Reranking model name ====> BAAI/bge-reranker-base
========evaluate_trace=======
judge_model used ====> llama-3.3-70b-versatile
⚠️ Failed on question 0: what is the rate of return in cadence design systems inc . of an investment from... -> Judge output could not be parsed as JSON


KeyboardInterrupt: 

In [ ]:
#------ Full RAG Usage with metrics

config = {
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "chunk_size": 1024,
    "reranking": True,
    "rerank_model": "BAAI/bge-reranker-base",
}

row_idx=13
question = dataset["train"][row_idx]["question"]

row = evaluate_and_log(
    question=question,
    dataset_train=dataset["train"],
    dataset_row_index=row_idx,
    strategy="no_chunk",
    config=config,
    k=10,
    llm_model="qwen/qwen3.6-27b",
    judge_model="llama-prompt-guard-2-86m",
    csv_path = "rag_finqa_RMSE1_runs.csv",
)

print(row)


NameError: name 'evaluate_and_log' is not defined

In [ ]:
print(len(dataset["test"]["question"]))

2294


In [ ]:
#--------- No-chunking: embed whole documents as single chunks ----------
def no_chunk_documents(documents):
  """
  Treats each full document as one chunk (no splitting).
  Mirrors the exact output shape of chunk_documents(), so it plugs
  directly into flatten_chunks() / embed_texts() / build_faiss_index() /
  save_index() with zero changes to those functions.
  """
  return [
      {"doc_index": i, "chunks": [" ".join(doc).strip()]}
      for i, doc in enumerate(documents)
      if "".join(doc).strip()
  ]

no_chunk_results = no_chunk_documents(documents)
print(no_chunk_results[0])

# Same downstream flow as your other strategies
single = {"no_chunk": no_chunk_results}
texts, metas = flatten_chunks(single)

print(f"\n==== Embedding Strategy: no_chunk ({len(texts)} chunks) ====")
embeddings, model = embed_texts(texts)
index = build_faiss_index(embeddings)
save_index(index, metas, texts, "no_chunk", out_dir="/content/drive/MyDrive/faiss_store")

In [ ]:
#--------- calling the llm with question ---------

result = answer_with_groq(
    query="what was the percent of the decrease in the other intangible assets net from 2003 to 2004?",
    strategy = "table",
    k=5,
    model=llm_model,
    rerank=bool(config.get("reranking", False)),
    rerank_candidates=config.get("rerank_candidates", 10),)

print("============  Answer ===========")
print(result["answer"])

In [ ]:
#------ search usage ------------------

hits = search(
    query="what is the ultimate health care trend rate?",
    strategy = "table",
    k=5,
)

for h in hits:
  print(f"[{h['score']:.3f}] (d{h['doc_index']} c{h['chunk_index']})")
  print(h["text"][:200], "\n-----------")


In [ ]:
#--------- Sample usage to test chunking strategy
from os import name
configs = {
    "fixed": {"strategy": "fixed", "params": {"chunk_size": 1024}},
    "recursive": {"strategy": "recursive", "params": {"chunk_size": 1024, "chunk_overlap": 50}},
    "sentence": {"strategy": "sentence", "params": {"sentences_per_chunk": 5}},
    "table": {"strategy": "table", "params": {"chunk_size": 1024}},
    "sliding": {"strategy": "sliding", "params": {"window_size": 1024, "step": 250}},
    "structure": {"strategy": "structure", "params": {"max_chunk_size": 800}}
}

results = chunk_documents(documents, configs["recursive"])
print(results[0])

stores = {}

for strategy_name, docs in {"table": results}.items():
  single = {strategy_name: docs}
  texts, metas = flatten_chunks(single)
  if not texts:
    continue
  print(f"\n==== Embedding Strategy: {strategy_name} ({len(texts)} chunks) ====")
  embeddings, model = embed_texts(texts)
  index = build_faiss_index(embeddings)
  save_index(index, metas, texts, strategy_name)
  stores[strategy_name] = {"index": index, "metas": metas,
                           "texts": texts, "model": model}